# 02 - Register a historical backfill partition

Bulk-loads manifest version 1 files into the same `video_work` queue used by event intake. Partition the inventory by date/prefix and run one bounded registration activity per partition. This notebook registers work only; the normal dispatcher controls inference concurrency.

**After importing into Fabric:** On the configuration code cell, select **... -> Toggle parameter cell** and confirm the parameter indicator. Then attach and pin `people_counter_<environment>` as this notebook's default Lakehouse.

In [ ]:
MANIFEST_GLOB = ""
REGISTRATION_ID = ""
SOURCE_STORAGE_ACCOUNT = ""
SOURCE_CONTAINER = ""
SOURCE_SHORTCUT_NAME = ""
DATABASE = ""
TABLE_PREFIX = "people_counter"
MAX_ATTEMPTS = 4
PRIORITY = 10
PIPELINE = "rtdetr-osnet"
DEVICE_VARIANT = "cpu"
DEVICE = "cpu"
BATCH_SIZE = 1
SAMPLE_FPS = 3.0
DETECTION_THRESHOLD = 0.6
USE_FP16 = False
DETECTOR_MODEL = "r18"
CAMERA_MOTION_COMPENSATION = None

In [ ]:
from datetime import datetime, timedelta, timezone
from urllib.parse import unquote_to_bytes, urlsplit, urlunsplit
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError
import json
import re
import uuid

from people_counter.fabric_control import ControlWriter
import notebookutils
from pyspark.sql import SparkSession, functions as F


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
ENCODED_SEPARATOR = re.compile(r"%2f|%5c", re.IGNORECASE)
database = DATABASE.strip()
prefix = TABLE_PREFIX.strip()
if database and IDENTIFIER.fullmatch(database) is None:
    raise ValueError("DATABASE is not a valid identifier")
if IDENTIFIER.fullmatch(prefix) is None:
    raise ValueError("TABLE_PREFIX is not a valid identifier")
manifest_glob = MANIFEST_GLOB.strip()
manifest_glob_parts = urlsplit(manifest_glob)
if manifest_glob_parts.scheme.lower() != "abfss" or manifest_glob_parts.hostname != "onelake.dfs.fabric.microsoft.com":
    raise ValueError("MANIFEST_GLOB must be an absolute OneLake ABFSS shortcut path")
registration_id = REGISTRATION_ID.strip()
if not registration_id:
    raise ValueError("REGISTRATION_ID must be the backfill pipeline run ID")
max_attempts = int(MAX_ATTEMPTS)
if max_attempts < 1:
    raise ValueError("MAX_ATTEMPTS must be at least 1")
for parameter_name, parameter_value in (
    ("SOURCE_STORAGE_ACCOUNT", SOURCE_STORAGE_ACCOUNT),
    ("SOURCE_CONTAINER", SOURCE_CONTAINER),
    ("SOURCE_SHORTCUT_NAME", SOURCE_SHORTCUT_NAME),
):
    if not isinstance(parameter_value, str) or not parameter_value.strip():
        raise ValueError(f"{parameter_name} must be a non-empty string")


def table(suffix: str) -> str:
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


def parse_bool(value: object, name: str) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, str) and value.strip().lower() in {"true", "false"}:
        return value.strip().lower() == "true"
    raise ValueError(f"{name} must be true or false")


def parse_optional_bool(value: object, name: str) -> bool | None:
    if value is None or (isinstance(value, str) and value.strip().lower() in {"", "null", "none"}):
        return None
    return parse_bool(value, name)


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
writer = ControlWriter(spark_session, table("control_writer"))
DeltaTable = writer.tables
work_table = table("video_work")
registration_leases_table = table("registration_leases")
if PIPELINE not in {"rtdetr-osnet", "rfdetr-botsort"}:
    raise ValueError("PIPELINE must be rtdetr-osnet or rfdetr-botsort")
if DEVICE_VARIANT != "cpu":
    raise ValueError("Fabric-native Spark requires DEVICE_VARIANT=cpu")
if DEVICE != "cpu":
    raise ValueError("Fabric-native Spark requires DEVICE=cpu")
validated_batch_size = int(BATCH_SIZE)
validated_sample_fps = None if SAMPLE_FPS is None else float(SAMPLE_FPS)
validated_threshold = float(DETECTION_THRESHOLD)
if validated_batch_size < 1 or (validated_sample_fps is not None and validated_sample_fps <= 0):
    raise ValueError("BATCH_SIZE and SAMPLE_FPS are invalid")
if not 0.1 <= validated_threshold <= 1:
    raise ValueError("DETECTION_THRESHOLD must be between 0.1 and 1")
validated_use_fp16 = parse_bool(USE_FP16, "USE_FP16")
if validated_use_fp16:
    raise ValueError("Fabric-native CPU execution requires USE_FP16=false")
if DETECTOR_MODEL not in {"r18", "r50"}:
    raise ValueError("DETECTOR_MODEL must be r18 or r50")
validated_camera_motion = parse_optional_bool(CAMERA_MOTION_COMPENSATION, "CAMERA_MOTION_COMPENSATION")


def normalize_uri(value: str) -> str:
    if not isinstance(value, str) or not value:
        return ""
    parts = urlsplit(value)
    if parts.query or parts.fragment or ENCODED_SEPARATOR.search(parts.path):
        return ""
    try:
        decoded_path = unquote_to_bytes(parts.path).decode("utf-8", "strict")
    except UnicodeDecodeError:
        return ""
    segments = decoded_path.split("/")[1:]
    if not segments or any(
        segment in {"", ".", ".."}
        or "\\" in segment
        or "\x00" in segment
        or "%" in segment
        for segment in segments
    ):
        return ""
    account = SOURCE_STORAGE_ACCOUNT.strip().lower()
    container = SOURCE_CONTAINER.strip()
    shortcut_name = SOURCE_SHORTCUT_NAME.strip()
    if parts.scheme.lower() == "abfss" and parts.hostname == "onelake.dfs.fabric.microsoft.com":
        marker = ["Files", shortcut_name]
        marker_index = next(
            (
                index
                for index in range(len(segments) - 1)
                if segments[index : index + 2] == marker
            ),
            None,
        )
        if marker_index is None or marker_index + 2 >= len(segments):
            return ""
        relative = segments[marker_index + 2 :]
        return f"abfss://{container}@{account}.dfs.core.windows.net/{'/'.join(relative)}"
    if parts.scheme.lower() == "https" and parts.hostname in {
        f"{account}.dfs.core.windows.net",
        f"{account}.blob.core.windows.net",
    }:
        if parts.username or parts.password or parts.port or segments[0] != container:
            return ""
        return f"abfss://{container}@{account}.dfs.core.windows.net/{'/'.join(segments[1:])}"
    expected_netloc = f"{container}@{account}.dfs.core.windows.net"
    if parts.scheme.lower() not in {"abfs", "abfss"} or parts.netloc.lower() != expected_netloc:
        return ""
    path = "/" + "/".join(segments)
    return urlunsplit((parts.scheme.lower(), expected_netloc, path, "", ""))


def valid_timezone(value: str) -> bool:
    try:
        ZoneInfo(value)
    except (TypeError, ZoneInfoNotFoundError):
        return False
    return True


def config_for_line(line: list[int]) -> str:
    return json.dumps(
        {
            "pipeline": PIPELINE,
            "device_variant": DEVICE_VARIANT,
            "device": DEVICE,
            "batch_size": validated_batch_size,
            "sample_fps": validated_sample_fps,
            "detection_threshold": validated_threshold,
            "use_fp16": validated_use_fp16,
            "line": line,
            "detector_model": DETECTOR_MODEL,
            "camera_motion_compensation": validated_camera_motion,
        },
        sort_keys=True,
        separators=(",", ":"),
    )


normalize_uri_udf = F.udf(normalize_uri, "string")
valid_timezone_udf = F.udf(valid_timezone, "boolean")
config_for_line_udf = F.udf(config_for_line, "string")
now = datetime.now(timezone.utc)
raw = spark_session.read.option("multiLine", "true").json(manifest_glob).withColumn(
    "manifest_uri",
    F.input_file_name(),
)
required = {
    "schema_version",
    "asset_id",
    "asset_version",
    "video_uri",
    "source_etag",
    "expected_size_bytes",
    "expected_sha256",
    "camera_id",
    "location_id",
    "captured_at_utc",
    "camera_timezone",
    "counting_line",
}
missing_columns = sorted(required - set(raw.columns))
if missing_columns:
    raise ValueError(f"Backfill manifests are missing columns: {missing_columns}")
if "duration_seconds" not in raw.columns:
    raw = raw.withColumn("duration_seconds", F.lit(None).cast("double"))

raw = (
    raw.withColumn("_source_uri", normalize_uri_udf("video_uri"))
    .withColumn("_manifest_uri", normalize_uri_udf("manifest_uri"))
    .withColumn("_captured_at_utc", F.to_timestamp("captured_at_utc"))
    .withColumn("_counting_line", F.transform("counting_line", lambda value: value.cast("int")))
    .withColumn("_config_json", config_for_line_udf("_counting_line"))
)
blank_required = None
for column in (
    "asset_id", "asset_version", "video_uri", "source_etag",
    "expected_sha256", "camera_id", "location_id", "captured_at_utc",
    "camera_timezone",
):
    check = F.col(column).isNull() | (F.trim(F.col(column).cast("string")) == "")
    blank_required = check if blank_required is None else blank_required | check
invalid = raw.where(
    F.col("schema_version").isNull()
    | (F.col("schema_version") != 1)
    | blank_required
    | (F.col("_source_uri") == "")
    | ~F.col("_source_uri").contains("/incoming/")
    | (F.col("_manifest_uri") == "")
    | ~F.col("_manifest_uri").contains("/incoming/")
    | ~F.col("_manifest_uri").endswith(".json")
    | F.col("source_etag").isNull()
    | F.col("expected_size_bytes").isNull()
    | ~F.trim(F.col("expected_size_bytes").cast("string")).rlike("^[0-9]+$")
    | F.col("expected_size_bytes").cast("long").isNull()
    | F.col("expected_size_bytes").cast("double").isNull()
    | (F.col("expected_size_bytes").cast("long") <= 0)
    | (F.col("expected_size_bytes").cast("double") != F.col("expected_size_bytes").cast("long").cast("double"))
    | F.col("expected_sha256").isNull()
    | ~F.lower("expected_sha256").rlike("^[0-9a-f]{64}$")
    | F.col("camera_id").isNull()
    | F.col("location_id").isNull()
    | F.col("_captured_at_utc").isNull()
    | ~F.col("captured_at_utc").cast("string").rlike("(?:Z|[+-][0-9]{2}:[0-9]{2})$")
    | F.col("camera_timezone").isNull()
    | ~valid_timezone_udf("camera_timezone")
    | F.col("counting_line").isNull()
    | (F.size("counting_line") != 4)
    | F.exists(
        "counting_line",
        lambda value: value.cast("double").isNull()
        | (value.cast("double") != value.cast("int").cast("double")),
    )
    | (
        F.col("duration_seconds").isNotNull()
        & (
            F.col("duration_seconds").cast("double").isNull()
            | F.isnan(F.col("duration_seconds").cast("double"))
            | (F.col("duration_seconds").cast("double") <= 0)
            | (F.abs(F.col("duration_seconds").cast("double")) > F.lit(1.7976931348623157e308))
        )
    )
)
invalid_count = invalid.count()
if invalid_count:
    examples = [row.manifest_uri for row in invalid.select("manifest_uri").limit(20).collect()]
    raise ValueError(f"Rejected {invalid_count} invalid manifests; examples={examples}")

source = raw.select(
    F.sha2(F.concat_ws("\n", F.col("_source_uri"), F.col("asset_version")), 256).alias("work_id"),
    "asset_id",
    "asset_version",
    F.col("_source_uri").alias("source_uri"),
    F.col("_manifest_uri").alias("manifest_uri"),
    "source_etag",
    F.col("expected_size_bytes").cast("long"),
    F.lower("expected_sha256").alias("expected_sha256"),
    "camera_id",
    "location_id",
    F.col("_captured_at_utc").alias("captured_at_utc"),
    "camera_timezone",
    F.col("duration_seconds").cast("double"),
    F.lit(int(PRIORITY)).cast("int").alias("priority"),
    F.lit("QUEUED").alias("status"),
    F.lit(now).alias("received_at"),
    F.lit(now).alias("queued_at"),
    F.lit(now).alias("queue_entered_at"),
    F.lit(None).cast("timestamp").alias("not_before_at"),
    F.lit(0).cast("int").alias("attempt_count"),
    F.lit(max_attempts).cast("int").alias("max_attempts"),
    F.lit(None).cast("string").alias("lease_owner_attempt_id"),
    F.lit(None).cast("string").alias("lease_dispatcher_id"),
    F.lit(None).cast("timestamp").alias("lease_acquired_at"),
    F.lit(None).cast("timestamp").alias("lease_expires_at"),
    F.lit(None).cast("timestamp").alias("last_heartbeat_at"),
    F.lit(None).cast("string").alias("committed_attempt_id"),
    F.lit(None).cast("timestamp").alias("completed_at"),
    F.lit(None).cast("string").alias("last_error_category"),
    F.lit(None).cast("string").alias("last_error_type"),
    F.lit(None).cast("string").alias("last_error_message"),
    F.lit(None).cast("string").alias("last_replay_id"),
    F.lit(0).cast("long").alias("replay_generation"),
    F.col("_config_json").alias("config_json"),
    F.sha2(F.col("_config_json"), 256).alias("config_sha256"),
    F.to_date("_captured_at_utc").alias("capture_date"),
)
immutable_fields = [
    "asset_id", "asset_version", "source_uri", "manifest_uri",
    "source_etag", "expected_size_bytes", "expected_sha256",
    "camera_id", "location_id", "captured_at_utc", "camera_timezone",
    "duration_seconds", "config_sha256", "capture_date",
]
conflicting_work = (
    source.withColumn(
        "manifest_fingerprint",
        F.sha2(F.to_json(F.struct(*immutable_fields)), 256),
    )
    .groupBy("work_id")
    .agg(F.countDistinct("manifest_fingerprint").alias("variants"))
    .where(F.col("variants") > 1)
)
if conflicting_work.limit(1).count():
    examples = [row.work_id for row in conflicting_work.select("work_id").limit(20).collect()]
    raise ValueError(f"Conflicting manifests resolve to the same work_id: {examples}")
source = source.dropDuplicates(["work_id"])
source = source.cache()
source_count = source.count()
lock_owner = f"{registration_id}:{uuid.uuid4().hex}"
lock_source = spark_session.createDataFrame(
    [("global", lock_owner, now, now + timedelta(minutes=15))],
    "lock_name string, owner_id string, acquired_at timestamp, expires_at timestamp",
)
(
    DeltaTable.forName(spark_session, registration_leases_table)
    .alias("t")
    .merge(lock_source.alias("s"), "t.lock_name = s.lock_name")
    .whenMatchedUpdateAll(condition="t.expires_at <= current_timestamp()")
    .execute()
)
def assert_registration_lock() -> None:
    lock_rows = (
        spark_session.table(registration_leases_table)
        .where(
            (F.col("lock_name") == "global")
            & (F.col("owner_id") == lock_owner)
            & (F.col("expires_at") > F.current_timestamp())
        )
        .limit(2)
        .collect()
    )
    if len(lock_rows) != 1:
        raise RuntimeError("Registration mutex ownership was lost or expired")


def renew_registration_lock() -> None:
    renewed_until = datetime.now(timezone.utc) + timedelta(minutes=15)
    DeltaTable.forName(spark_session, registration_leases_table).update(
        condition=(
            (F.col("lock_name") == "global")
            & (F.col("owner_id") == lock_owner)
            & (F.col("expires_at") > F.current_timestamp())
        ),
        set={"expires_at": F.lit(renewed_until)},
    )
    assert_registration_lock()


def release_registration_lock() -> None:
    DeltaTable.forName(spark_session, registration_leases_table).update(
        condition=(F.col("lock_name") == "global") & (F.col("owner_id") == lock_owner),
        set={"owner_id": F.lit(""), "expires_at": F.lit(datetime.now(timezone.utc))},
    )


def run_with_lock_cleanup(operation):
    try:
        return operation()
    except Exception:
        release_registration_lock()
        source.unpersist()
        raise


try:
    assert_registration_lock()
except Exception:
    release_registration_lock()
    source.unpersist()
    raise RuntimeError("Another intake activity owns the registration mutex")


existing_work = spark_session.table(work_table).select("work_id", *immutable_fields)
run_with_lock_cleanup(renew_registration_lock)
same_immutable = None
for field in immutable_fields:
    check = F.col(f"s.{field}").eqNullSafe(F.col(f"t.{field}"))
    same_immutable = check if same_immutable is None else same_immutable & check
existing_conflicts = (
    source.alias("s")
    .join(existing_work.alias("t"), F.col("s.work_id") == F.col("t.work_id"), "inner")
    .where(~same_immutable)
    .select(F.col("s.work_id").alias("work_id"))
)
existing_conflict_rows = run_with_lock_cleanup(
    lambda: existing_conflicts.limit(20).collect()
)
if existing_conflict_rows:
    examples = [row.work_id for row in existing_conflict_rows]
    release_registration_lock()
    source.unpersist()
    raise ValueError(f"Backfill manifests conflict with registered work: {examples}")
before_count = run_with_lock_cleanup(
    lambda: spark_session.table(work_table).join(source.select("work_id"), "work_id", "inner").count()
)
run_with_lock_cleanup(renew_registration_lock)
run_with_lock_cleanup(
    lambda: (
        DeltaTable.forName(spark_session, work_table)
        .alias("t")
        .merge(source.alias("s"), "t.work_id = s.work_id")
        .whenNotMatchedInsertAll()
        .execute()
    )
)
run_with_lock_cleanup(assert_registration_lock)
persisted_work = spark_session.table(work_table).select("work_id", *immutable_fields)
persisted_conflicts = (
    source.alias("s")
    .join(persisted_work.alias("t"), F.col("s.work_id") == F.col("t.work_id"), "inner")
    .where(~same_immutable)
    .select(F.col("s.work_id").alias("work_id"))
)
persisted_conflict_rows = run_with_lock_cleanup(
    lambda: persisted_conflicts.limit(20).collect()
)
if persisted_conflict_rows:
    examples = [row.work_id for row in persisted_conflict_rows]
    release_registration_lock()
    source.unpersist()
    raise ValueError(f"Persisted work conflicts with backfill manifests: {examples}")
persisted_count = run_with_lock_cleanup(
    lambda: spark_session.table(work_table).join(source.select("work_id"), "work_id", "inner").count()
)
if persisted_count != source_count:
    release_registration_lock()
    source.unpersist()
    raise RuntimeError(f"Expected {source_count} registered work rows, found {persisted_count}")
release_registration_lock()
source.unpersist()
outcome = {
    "manifest_count": source_count,
    "already_registered": before_count,
    "newly_registered": source_count - before_count,
    "manifest_glob": manifest_glob,
}
print(json.dumps(outcome, sort_keys=True))

In [ ]:
notebookutils.notebook.exit(json.dumps(outcome, sort_keys=True))